<!-- paper-paired-role-banner -->
> 🟦 **실습 코드 · 왼쪽 창** — TODO를 먼저 직접 구현하세요. 막히면 오른쪽 정답의 같은 셀에서 필요한 한 줄만 확인합니다.

# 18. DDPM — 확산과 역확산 — Denoising Diffusion Probabilistic Models

- **원 논문:** [Denoising Diffusion Probabilistic Models](https://arxiv.org/abs/2006.11239)
- **저자 / 발표:** Jonathan Ho, Ajay Jain, Pieter Abbeel · NeurIPS (2020)
- **난이도 / 예상 시간:** 중상급 · 약 90분
- **선수 지식:** 정규분포, 재매개화, MSE, 조건부 생성

이 노트북은 논문 결과 수치를 그대로 재현하는 대규모 benchmark가 아니라,
핵심 수식과 구조가 실제로 어떻게 동작하는지 확인하는 **하드웨어 인식 미니 재현**입니다.
PyTorch 학습은 CUDA → Apple MPS → CPU 순으로 자동 선택하며, 작은 NumPy·Python 실험은
전송 비용을 피하기 위해 CPU에서 실행될 수 있습니다.

## 재현 범위와 완료 기준

**이 노트북이 재현하는 것**

식 (4)의 임의 시점 forward noising, 식 (11)의 noise parameterization, 식 (14)와 Algorithm 1의 noise-prediction 학습, Algorithm 2의 reverse sampling을 2차원 두 군집에 구현하고 시간에 따른 분포 변화를 그립니다.

**원 논문과 다른 것**

CIFAR-10/LSUN 영상, U-Net, 1,000 diffusion step과 FID/Inception Score는 생략하고, 2차원 데이터·작은 MLP·30 step을 씁니다. 그림 품질이나 논문의 benchmark 수치는 재현 대상이 아닙니다.

완료하려면 TODO를 구현하고, 검증 셀의 `assert`를 모두 통과시키고, 마지막 관찰 질문에
자신의 말로 답하세요. 수치가 논문과 다르더라도 핵심 불변식과 비교 방향이 맞으면 성공입니다.

## 논문 ↔ 노트북 지도

        절 번호와 수식 번호는 위 링크의 대표 공개본을 기준으로 합니다. 판본에 따라 페이지는 달라질 수
        있으므로 **절 제목과 수식 번호**를 함께 사용하세요.

        | 원 논문의 위치 | 이 노트북의 대응 실습 | 확인할 증거 |
        |---|---|---|
        | §2 식 (2), §3.2 | linear β schedule과 forward Markov 과정 | alpha 및 누적 alpha_bar |
| §2 식 (4) | 닫힌형식 q(x_t|x_0) | t=0 경계·표본 shape |
| §3.2 식 (11) | ε_θ로 reverse mean 계산 | 한 step 역확산 |
| §3.4 식 (14), Algorithm 1 | 무작위 t noise MSE | 초기/후반 loss |
| Algorithm 2 | T−1부터 0까지 sampling | trajectory 산점도 |

        > 읽기 순서: 먼저 해당 절을 훑고 → 코드를 예측해 작성하고 → 출력이 논문의 주장과 왜
        > 연결되는지 한 문장으로 설명합니다.

## 핵심 수식과 구현 설계

$$
q(x_t\mid x_0)=\mathcal{N}\!\left(
x_t;\sqrt{\bar\alpha_t}x_0,(1-\bar\alpha_t)I\right),\quad
\mathcal{L}=\mathbb{E}\|\epsilon-\epsilon_\theta(x_t,t)\|_2^2
$$

- $\bar\alpha_t$는 누적 noise schedule, $\epsilon_\theta$는 timestep 조건부 noise predictor입니다.
- forward process는 닫힌 형태로 $x_t$를 만들고 reverse model은 주입된 noise를 예측합니다.
- schedule, timestep sampling, noising, predictor, denoising step을 각각 함수로 구현합니다.
- 이 축소 실험의 $x_0,x_t,\epsilon$ shape은 모두 $[B,2]$이며 timestep $[B]$가 조건으로 들어갑니다.

## 구현 선택 이유

- **원문에서 보존한 부분:** §2 식 (2), §3.2
- **노트북 구현:** linear β schedule과 forward Markov 과정
- **이렇게 구현한 이유:** 완성된 고수준 model을 한 줄로 호출하면
  논문의 핵심 shape·mask·loss·상태 변화가 숨겨집니다. 따라서 이 계산을
  이름이 드러나는 class와 함수로 나누고 중간 tensor를 assertion으로
  검사합니다.
- **실패를 잡는 증거:** alpha 및 누적 alpha_bar를 출력 크기만 아니라
  구현 불변식으로 사용합니다.
- **축소 선택:** CIFAR-10/LSUN 영상, U-Net, 1,000 diffusion step과 FID/Inception Score는 생략하고, 2차원 데이터·작은 MLP·30 step을 씁니다. 그림 품질이나 논문의 benchmark 수치는 재현 대상이 아닙니다.는 기본 실행에서 생략합니다.
  이는 Windows CPU에서도 반복 실험이 가능하게 하면서 핵심 메커니즘은
  유지하기 위한 선택입니다.

### 구현 단계 1 · 실행 환경·데이터 계약 (setup 예외)

- **원문 위치:** 해당 없음 — import, seed, device, 로컬 데이터 로딩만 담당하며 논문의 학습 연산을 구현하지 않습니다.
- **Tensor shape 계약:** clean/noisy/noise [B,2] + timestep [B] → predicted noise [B,2]
- **구현 이유:** 이후 셀의 비교가 재현 가능하도록 장치·dtype·seed와 입력 불변식을 먼저 고정합니다.
- **완료 증거:** 데이터 존재 여부, 입력 rank, 장치 선택 assertion과 환경 요약 출력이 통과해야 합니다.

In [ ]:
import math
import random

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import Tensor, nn
from torch.nn import functional as F
from llm_engineering_lab.acceleration import get_accelerator

random.seed(0)
np.random.seed(0)
torch.manual_seed(0)
ACCELERATOR = get_accelerator()
DEVICE = ACCELERATOR.device
if DEVICE.type == "cuda":
    torch.cuda.manual_seed_all(0)
print(ACCELERATOR.summary())
print("synthetic-data preprocessing stays on CPU; neural training uses DEVICE")

## 1) 식 (4): 한 번에 원하는 시점으로

`α_t=1-β_t`, `ᾱ_t=∏_{s≤t} α_s`이면 반복해서 잡음을 넣지 않고도
`x_t = sqrt(ᾱ_t)x_0 + sqrt(1-ᾱ_t)ε`로 직접 표본화할 수 있습니다.

### 구현 단계 2 · 핵심 연산·모델

- **원문 위치:** §2 식 (2), §3.2 / §2 식 (4)
- **이 셀의 책임:** linear β schedule과 forward Markov 과정 / 닫힌형식 q(x_t|x_0)
- **Tensor shape 계약:** clean/noisy/noise [B,2] + timestep [B] → predicted noise [B,2]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** alpha 및 누적 alpha_bar / t=0 경계·표본 shape. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
# TODO 18-1: schedule과 식 (4)의 q_sample을 구현하세요.
T = 30
betas = ...
alphas = ...
alpha_bar = ...


def extract(values: Tensor, t: Tensor, x: Tensor) -> Tensor:
    """실습 계약: 정답과 같은 공개 signature를 유지한다.
    원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
    result = None
    assert result is not None, "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
    return result


def q_sample(
    x0: Tensor, t: Tensor, noise: Tensor | None = None
) -> tuple[Tensor, Tensor]:
    """실습 계약: 정답과 같은 공개 signature를 유지한다.
    원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
    result = None
    assert result is not None, "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
    return result


def _return_shape_fixture(result):
    """바로 위 Markdown의 반환 shape 계약을 확인한다."""
    assert result is not None, "TODO 결과가 아직 비어 있습니다."
    shape = getattr(result, "shape", None)
    container = isinstance(result, (tuple, list, dict, float, int))
    assert shape is not None or container
    return result

### 구현 단계 3 · 논문 주장 확인 실험

- **원문 위치:** §2 식 (2), §3.2 / §2 식 (4)
- **이 셀의 책임:** linear β schedule과 forward Markov 과정 / 닫힌형식 q(x_t|x_0)
- **Tensor shape 계약:** clean/noisy/noise [B,2] + timestep [B] → predicted noise [B,2]
- **구현 이유:** 한 변수만 바꾸어 해당 메커니즘이 출력에 미치는 영향을 분리합니다.
- **완료 증거:** alpha 및 누적 alpha_bar / t=0 경계·표본 shape. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
# 준비: 외부 다운로드 없는 2차원 두 군집
g = torch.Generator().manual_seed(18)
half = 256
data = torch.cat(
    [
        torch.randn(half, 2, generator=g) * torch.tensor([0.35, 0.18])
        + torch.tensor([-1.5, 0.0]),
        torch.randn(half, 2, generator=g) * torch.tensor([0.35, 0.18])
        + torch.tensor([1.5, 0.0]),
    ]
)
times = torch.tensor([0, T // 3, 2 * T // 3, T - 1])
fig, axes = plt.subplots(1, 4, figsize=(10, 2.4))
for axis, time in zip(axes, times):
    xt, _ = q_sample(data, torch.full((len(data),), int(time)))
    axis.scatter(xt[:, 0], xt[:, 1], s=4, alpha=0.35)
    axis.set_title(f"t={int(time)}")
    axis.set_xlim(-4, 4)
    axis.set_ylim(-3, 3)
plt.tight_layout()
plt.show()

### 구현 단계 4 · 핵심 연산·모델

- **원문 위치:** §3.2 식 (11)
- **이 셀의 책임:** ε_θ로 reverse mean 계산
- **Tensor shape 계약:** clean/noisy/noise [B,2] + timestep [B] → predicted noise [B,2]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** 한 step 역확산. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
# TODO 18-2: timestep embedding과 ε_θ(x_t,t) MLP를 구현하세요.
def time_embedding(t: Tensor, dim: int = 16) -> Tensor:
    """실습 계약: 정답과 같은 공개 signature를 유지한다.
    원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
    result = None
    assert result is not None, "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
    return result


class NoisePredictor(nn.Module):
    """실습 계약: 정답과 같은 공개 signature를 유지한다.
    원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""

    def __init__(self):
        """실습 계약: 정답과 같은 공개 signature를 유지한다.
        원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
        self._exercise_contract_pending = True

    def forward(self, x: Tensor, t: Tensor) -> Tensor:
        """실습 계약: 정답과 같은 공개 signature를 유지한다.
        원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
        result = None
        assert result is not None, (
            "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
        )
        return result


def _return_shape_fixture(result):
    """바로 위 Markdown의 반환 shape 계약을 확인한다."""
    assert result is not None, "TODO 결과가 아직 비어 있습니다."
    shape = getattr(result, "shape", None)
    container = isinstance(result, (tuple, list, dict, float, int))
    assert shape is not None or container
    return result

### 구현 단계 5 · 목적함수·학습 update

- **원문 위치:** §3.4 식 (14), Algorithm 1
- **이 셀의 책임:** 무작위 t noise MSE
- **Tensor shape 계약:** clean/noisy/noise [B,2] + timestep [B] → predicted noise [B,2]
- **구현 이유:** forward 결과와 논문 objective, gradient update의 책임을 분리합니다.
- **완료 증거:** 초기/후반 loss. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
# TODO 18-3: Algorithm 1 / 식 (14)의 random-t noise MSE로 학습하세요.
predictor = ACCELERATOR.move(NoisePredictor())
data, betas, alphas, alpha_bar = ACCELERATOR.move(data, betas, alphas, alpha_bar)
optimizer = torch.optim.Adam(predictor.parameters(), lr=2e-3)
losses = []
_stage_result = None
assert _stage_result is not None, "TODO: 위 단계의 결과와 검증을 구현하세요."

### 구현 단계 6 · 핵심 연산·모델

- **원문 위치:** §3.2 식 (11) / Algorithm 2
- **이 셀의 책임:** ε_θ로 reverse mean 계산 / T−1부터 0까지 sampling
- **Tensor shape 계약:** clean/noisy/noise [B,2] + timestep [B] → predicted noise [B,2]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** 한 step 역확산 / trajectory 산점도. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
# TODO 18-4: 식 (11)과 Algorithm 2의 reverse step/sampling을 구현하세요.
@torch.no_grad()
def reverse_step(model: nn.Module, x: Tensor, step: int) -> Tensor:
    """실습 계약: 정답과 같은 공개 signature를 유지한다.
    원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
    result = None
    assert result is not None, "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
    return result


@torch.no_grad()
def sample(model: nn.Module, count: int = 512) -> tuple[Tensor, list[Tensor]]:
    """실습 계약: 정답과 같은 공개 signature를 유지한다.
    원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
    result = None
    assert result is not None, "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
    return result


def _return_shape_fixture(result):
    """바로 위 Markdown의 반환 shape 계약을 확인한다."""
    assert result is not None, "TODO 결과가 아직 비어 있습니다."
    shape = getattr(result, "shape", None)
    container = isinstance(result, (tuple, list, dict, float, int))
    assert shape is not None or container
    return result

## 마무리 관찰 기록

아래 세 문장을 직접 완성하세요.

1. 이 논문의 핵심 연산은 `_____`이고, 코드에서는 `_____`에 해당한다.
2. 원 규모 실험 대신 미니 재현을 사용했기 때문에 확인할 수 없는 주장은 `_____`이다.
3. 한 가지 변수를 바꾸어 다시 실행한다면 `_____`를 바꾸고 `_____`를 측정하겠다.

**추가 실험:** seed는 유지한 채 한 변수만 바꾸고, 변경 전후의 metric을 표로 남기세요.